# Exercise Sheet 09

This markdown block is used to define macros for later markdown blocks. 
$\newcommand{\normaldist}{\mathcal{N}}$
$\newcommand{\betadist}{\text{Beta}}$
$\newcommand{\reals}{\mathbb{R}}$
$\newcommand{\ML}{\text{ML}}$
$\renewcommand{\vec}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\matrix}[1]{\boldsymbol{\mathbf{#1}}}$
$\newcommand{\dataset}{\mathcal{D}}$
$\newcommand{\class}{\mathcal{C}}$
$\newcommand{\optimal}[1]{#1^{\star}}$
$\newcommand{\argmin}{\text{argmin}}$
$\newcommand{\argmax}{\text{argmax}}$
$\newcommand{\expct}{\mathbb{E}}$
$\newcommand{\entropy}[1]{H[#1]}$
$\newcommand{\conditionalentropy}[2]{\entropy{#1 | #2}}$
$\newcommand{\kldiv}[2]{KL(#1 || #2)}$
$\newcommand{\mutualinfo}[2]{I[#1;#2]}$
$\newcommand{\deriv}[2]{\frac{d}{d #2} \left( #1\right)}$
$\newcommand{\inputs}{\matrix{X}}$
$\newcommand{\identitymtx}{\matrix{I}}$
$\newcommand{\designmtx}{\matrix{\Phi}}$
$\newcommand{\featurevec}{\boldsymbol{\phi}}$
$\newcommand{\weights}{\vec{w}}$
$\newcommand{\inputvec}{\vec{x}}$
$\newcommand{\norm}[1]{\lVert #1 \rVert}$
$\newcommand{\grad}{\nabla}$

In [ ]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import scipy.stats
import pandas as pd
from itertools import groupby

## Exercise 9.1 - Unsupervised Learning

Look back at the slides for Lecture 01 and, in particular at the slide entitled **Categorization in the Brain**. When first presented, you were told that one of the images was a 'tufa' and was asked which other images showed 'tufas'. Being able to generalise this label to other images is sometimes called $1$-shot learning - you only need 1 shot (or label) to learn a class - and it is theorised that your brain must have carried out some unsupervised learning in order to achieve this. Of the unsupervised methods described in Lecture 09's slide **Unsupervised Learning**, which do you think is most likely to be happening in the brain to facilitate this $1$ shot learning?

## Exercise 9.2 - K-means Clustering
This question investigates the clustering approaches discussed in the lecture, including K-means
and K-medoids.

Run the code block below which defines and calls the function `sample_and_fit_kmeans`. This should create a simple data-set using a mixture of Gaussians distribution and plot it coloured according to the originating components. Then it will fit K-means to the data, and replot it, colouring according to the discovered clusters. Do you notice any differences between the two plots?

Rerun the code a few times and see if the K-means algorithm consistently discovers the appropriate
clusters. Are the clusters always the same colour? Can you explain this?

In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

from fomlads.data.unsupervised import scenario1_data
from fomlads.model.clustering import kmeans
from fomlads.plot.unsupervised import scatter_plot_clusters

def sample_and_fit_kmeans():

    N=100
    datamtx, latent_components, means, covmtxs, mixcoefs = scenario1_data(N)
    fig, ax = scatter_plot_clusters(datamtx, latent_components, means)
    ax.set_title("Points indicating originating components")

    K = 3
    cluster_means, cluster_assignments = kmeans(datamtx, K)
    fig, ax = scatter_plot_clusters(
        datamtx, cluster_assignments, centres=cluster_means)
    ax.set_title("Points coloured with assigned cluster (K-means)")

sample_and_fit_kmeans()

## Exercise 9.3 - loss function $J$

### 9.3 a)
In the first code block below, a function `loss_function_progression_kmeans` has been started for you. Initially, this will simply plot a series of scatter plots, showing how the K-means algorithm progresses on the data. Inspect the code, including
the functions called in module `fomlads.model.clustering`, to see if you can understand how it
achieves this.

Now, complete the function `calculate_kmeans_loss` in module `fomlads.model.clustering` so that
it calculates the K-means loss function, J, from the lecture slides (slide title K-means). Once complete, you should call this function in each iteration of `loss_function_progression_kmeans` below, and collect the returned losses into a vector. After the iterations complete, you should not be able to  plot how the loss $J$ changes as the iteration number increases as part of the function `calculate_kmeans_loss`.

How do you expect $J$ to change? Can it ever decrease on an E-step? What about an M-step? Can it ever increase? 

Run the code a few times when finished to see how stable the K-means algorithm is.

### 9.3 b) [optional]

*You may choose to leave this until after watching all the videos for Unit 9.*

i. When using K-means on real data, one doesn’t always know in advance the best number
of clusters K to use. Instead, experimenters sometimes fit K-means to the data with
a selection of values K, and evaluate the loss J for each fitted model. Look in the second code block below, and complete the function `varying_k_kmeans` to do just that. You can use the function `plot_K_versus_loss` from module `fomlads.plot.unsupervised` to help visualise the results.

ii. The loss function J should be lower the larger a value K you choose, irrespective of the true number of clusters in your data. Look at the plot you produce, does J always decrease for larger K? If not, can you explain why? 

*Try fitting the algorithms a small number of times for each value of K, and plotting only
minimum value of J for each K. Is the plot better behaved? Why?*

### 9.3 c) [optional]

On the simple data used in **Ex. 9.3 b)**, you should see a point at K = 3 where the reduction in K begins to
plateau. However, it isn’t always clear from these graphs where that point is. You can explore
some of the other data-sets available in module `fomlads.data.unsupervised` from functions: `scenario2_data`, `scenario3_data` or `scenario4_data` instead of `scenario1_data`. Look at their implementation, and plot their
outputs, to help decide why these might be more difficult to select a good K for.

In the code block for **Ex. 9.3 b)**, import these new scenario functions and complete the function `varying_k_means` to support these other scenarios. In the third code block below, now call `varying_k_means` for these different scenarios and explore their features.

In [ ]:
from fomlads.data.wrangle import subsample_datapoints
from fomlads.model.clustering import calculate_kmeans_loss


def loss_function_progression_kmeans():
    """
    Plots the sequence of cluster assignments and a plot of the loss progression
    fitting K-means to a simple synthetic data-set
    """
    N=80
    K = 3
    iterations = 8
    losses = np.empty(iterations)
    datamtx, latent_components, means, covmtxs, mixcoefs = scenario1_data(N)
    centres = subsample_datapoints(datamtx, K)
    for iteration in range(iterations):
        centres, cluster_assignments = kmeans(
            datamtx, 3, initial_centres=centres, iterations=1)
        fig, ax = scatter_plot_clusters(
            datamtx, cluster_assignments, centres=centres)
        ax.set_title("Iteration %d" % iteration)
        ## TODO: 9.3 a) calculate the k-means loss here and store it

    ## TODO: 9.3 a) plot how the k-means loss changes with each iteration

loss_function_progression_kmeans()

In [ ]:
## code block for Ex. 9.3 b) and c)
from fomlads.plot.unsupervised import plot_K_versus_loss

## TODO: complete this function for Ex. 9.3 b) and 9.3 c)
def varying_k_kmeans(N, scenario=1):
    if scenario == 1:
        datamtx, _, _, _, _ = scenario1_data(N)
    else:
        raise ValueError("Unrecognised scenario for data")

## TODO: run the function with the appropriate arguments

In [ ]:
## Code block for the last part of 9.3 c)

## Exercise 9.4 - K-medoids


### 9.4 a)

Look at the function `sample_and_fit_kmedoids` in the code block below. This aims to perform a similar clustering procedure using just the proximity matrix generated from the data-matrix. The provided
code uses the function `kmedoids` from module `fomlads.model.clustering`. However, this relies
on two incomplete subfunctions in the same module: `kmedoids_e_step` and `kmedoids_m_step`.
Complete these functions to inplement the K-medoids algorithm. Then test your work by calling
`sample_and_fit_kmedoids`.

What do you notice about the location of the cluster centres, compared to the K-means approach?

What happens if you reduce the number of data-points on which the clustering is based?

### 9.4 b)

Imagine you had access to an anonymised data-set from a social network, in which users
are represented as integer ids and social connections as a pair of user ids, e.g. the connection
$(57, 83)$ would imply that user $57$ had a connection with user $83$. You wish to discover groups
(communities) of users which are highly connected with one another. Which of the algorithms
you saw in this week’s lecture would be most appropriate for this task? How could you represent
the data to achieve this?
How might you improve the performance of your clustering if you had demographic information
for each user too, e.g. age, gender, location, etc?

*There is no correct solution to this question, but you should decide on what you would try first given this problem. You can discuss it in the labs with the teaching staff if you wish to take it further.*

### 9.4 c) [optional]

*You can again ignore this until you have watched all the videos for Unit 9.*

Try plotting the loss $J$ for K-medoids for different values of $K$ just as you did for K-means. Are the results as you expect?


In [ ]:
## Run this code block after completing Ex. 9.4 a)
from fomlads.model.clustering import squared_euclidean_distance
from fomlads.model.clustering import kmedoids
from fomlads.model.clustering import calculate_kmedoids_loss

def sample_and_fit_kmedoids():

    N=100
    datamtx, latent_components, means, covmtxs, mixcoefs = scenario1_data(N)
    fig, ax = scatter_plot_clusters(datamtx, latent_components, means)
    ax.set_title("Points indicating originating components")

    K = 3
    proxmtx = squared_euclidean_distance(datamtx, datamtx)
    print("proxmtx[:5,:5] = %r" % (proxmtx[:5,:5],))
    cluster_medoids, cluster_assignments = kmedoids(proxmtx, K)
    cluster_means = datamtx[cluster_medoids,:]
    fig, ax = scatter_plot_clusters(
        datamtx, cluster_assignments, centres=cluster_means)
    ax.set_title("Points coloured with assigned cluster (K-means)")
    
sample_and_fit_kmedoids()

In [ ]:
## Code block for Ex. 9.4 c)

## Exercise 9.5 - Mixture of Gaussians

### 9.5 a)

Look at Result (9.4) on slide **Inspecting the MoG Model**.

i. Consider that you have $D$-dimensional data-points, i.e. $\vec{x}_n \in \reals^D$, and $K$ clusters, how many independent parameters do you have in your model in terms of $D$ and $K$? 

ii. If you constrained your clusters to have only diagonal elements in their covariance matrix, how does that change your answer to part i.? How does this decision affect the types of mixture distributions you can have?

iii. Imagine that $K=5$. How many parameters does your model from part i. have for $D=3$? What about $D=10$ or $D=100$? What about your model from part ii.?


## Exercise 9.6 - Mixture of Gaussians and EM
This question investigates the mixture of Gaussians model and EM algorithm from the lecture.

### 9.6 a)

Run `sample_and_fit_mog`, this should create a simple data-set using a mixture of Gaussians
distribution and plot it coloured according to the originating components. Then it will fit
a mixture of Gaussians to the data, and replot it, colouring points according to their component
responsibilities. How does the performance compare to the K-means algorithm on this same
data-set (scenario 3)?

### 9.6  b)
Look at the functions `em_mog`, `em_e_step` and `em_m_step` in module `fomlads.model.mog`. These are
for fitting the Mixture of Gaussians as described in the lectures. Which results in the lectures
slides do these correspond to? Some of the necessary values aren’t calculated directly, instead
the log of these values are calculated. Which values, and why do you think this is done?

### 9.6  c) 
Rather than the loss function J, a better way to evaluate a mixture of Gaussians is with a
negative joint log likelihood over the data-set. Result (9.4) tells you how to calculate a likelihood
value for a single data-point. Can you generalise this to define the likelihood of a collection of
data-points?

### 9.6 d)
When you have derived the mathematical form from **Ex. 9.6 c)**, complete the function
`log_likelihood_mog` in module `fomlads.model.mog` to calculate this for a data matrix and given
model parameters. Now implement a function in the code block below, `varying_k_mog`, that generates a single set of data, then fits a Mixture of Gaussians with the EM algorithm for a selection of Ks, plotting
the negative log likelihood against K on a graph, then call the function

Run the code block. What do you notice?

In [ ]:
from fomlads.model.mog import em_mog
from fomlads.plot.unsupervised import scatter_plot_responsibilities


def sample_and_fit_mog():
    """
    """
    N=100
    K=3
    datamtx, latent_components, true_means, covmtxs, mixcoefs = scenario3_data(N)

    fig, ax = scatter_plot_clusters(datamtx, latent_components, true_means)
    ax.set_title("Points indicating originating components")

    means, covmtxs, mixcoefs, log_resps = em_mog(datamtx, K)
    fig, ax = scatter_plot_responsibilities(
        datamtx, log_resps, means=means)
    ax.set_title("Points coloured with assigned cluster (K-means)")

sample_and_fit_mog()

In [ ]:
## Code block for Ex. 9.6 d)

## Exercise 9.7

As we have a probabilistic model, there are now more robust approaches we can use. One approach would be to define priors on all parameters and then to calculate the MAP or fully Bayesian solution before using this to select a model, but this is quite involved mathematically and beyond the scope of this course. An intermediate approach is to use an information criterion, which allows you to approximate Bayesian model selection without defining priors. There are two information criteria of interest here: Akaike's Information Criterion (AIC) (Akaike, 1974), and the Bayesian Information Criterion (Schwarz, 1978).

**AIC:** You can consider calculating the Akaike's information criterion as an alternative loss function. As well as the negative log likelihood of the data, this also includes a penality term for the number of parameters in your model. The intuition is that a more complex model, should fit the data better than a simpler model, and it is only worth using the more complex model if the improvement in the fit outweighs the increase in complexity.

The AIC is given by:
$$
AIC = 2M - 2 \ln (\hat{L})
$$
where $\hat{L}$ is your estimate of the maximum likelihood and $M$ is the total number of (independent) parameters in your model. For a MoG model with $K$ components in $D$ dimensional space, you will need: $K-1$ independent mixture components (the last is determined by the others), $KD$ parameters for the component means, and $KD(D-1)/2$ parameters for the covariance matrices. So the AIC becomes
$$AIC = K(D^2 + D + 2) - 2 - 2 \ln p(\inputs|\{\pi_{k},\vec{\mu}_{k}, \Sigma_{k}\})$$

You should now be able to find the best choice of $K$ by looking for the minimum in your model.

**BIC:** This is an alternative information criterion, which penalised complexity more highly. The recipe for the BIC is:

$$BIC = - 2 \ln (\hat{L}) + M(\ln (N) - \ln(2\pi))$$

where as before $\hat{L}$ is your estimate of the maximum likelihood for your model, $M$ is the number of parameters. Additionally, $N$ is the number of data-points in your model and $\pi$ is the mathematical constant.

Implement functions `aic_mog` and `bic_mog` in module `fomlads.model.mog` then use one or the other as a substitute for the negative log likelihood in your plots from `neg_log_likelihood`. You can now select the $K$ with the minimum AIC or BIC as your best choice.

In [ ]:
from fomlads.model.mog import log_likelihood_mog
from fomlads.model.mog import aic_mog
from fomlads.model.mog import bic_mog

## TODO: complete Ex. 9.7 here